# Анализ алгоритмов

In [1]:
import time
import random

import geopandas as gpd
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from genesis.metrics import *
from genesis.states import FirstArrivalUnitState
from genesis.utils import Progressbar, timing
from genesis.swiss_knife import CMAP_GrGldRd
from genesis.best_points import BestNodeHillClimbing, BestNodesFull, BestNodeMonkey, BestNodesHalfDiameter, BestNodeBee, NodeMetric
from genesis.mclp import BestNodesSA
from genesis.utils import TimeCheckPoint
from genesis.point_selectors import RandomNodesSelector
from genesis.stop_cases import NSameStopCase
from fire_units.metrics import Demand
from fire_units.mclp import BestNodesGAKopt
from genesis.mclp import BestNodesGA, BestNodesKoptG
from fire_units.best_points import BestNodeHillClimbingHD
from fire_units.metrics import CoverIndexBuilding
from genesis.stop_cases import MoreEqualStopCase

# Загрузка исходных данных и разбиение по районам

In [2]:
map_path = 'D:/QGIS/_Красноярск ДИССЕРТАЦИЯ'
# map_path = 'G:/QGIS/_Красноярск ДИССЕРТАЦИЯ'

# ГДС
G = ox.load_graphml(map_path+"/fire_data/rng.ml")
print(G.number_of_nodes(), G.number_of_edges())

# районы
districts = gpd.read_file(map_path+"/data/districts.gpkg", layer='Районы')
districts = districts.set_index(districts['name'])
# districts.plot(column='name', legend=True)
print('Районы Красноярска:', list(districts.index))

# здания
buildings = gpd.read_file(map_path+"/data/bld.gpkg", layer='здания')
buildings = buildings[['osmid', 'building', 'kfpo_spros', 'geometry']]   # 'name', 'amenity', 
buildings = buildings.set_index('osmid')
print('Всего зданий в Красноярске:', len(buildings))

31293 77344
Районы Красноярска: ['Железнодорожный район', 'Кировский район', 'Ленинский район', 'Октябрьский район', 'Свердловский район', 'Советский район', 'Центральный район']
Всего зданий в Красноярске: 58094


Установка скоростей следования для ГДС

In [3]:
from graphs.speeds import kmh_to_mm, set_graph_travel_times

speeds = [53, 40, 28.1, 17.3, 5.4]
speeds = [kmh_to_mm(s) for s in speeds]

set_graph_travel_times(G, speeds)

d:\Git\genesis\work\genesis\graphs\speeds.py:96: UserWarning: Граф был ранее упрощен! Оценка времени следования такого графа может повлечь неточности. Рекомендуется устанавливать время следования для неупрощенного графа и только после этого упрощать
  warnings.warn("Граф был ранее упрощен! Оценка времени следования такого графа может повлечь неточности. " + \


Приведение к единой системе координат

In [4]:
print('Проверка СК:', end=' ')
if not G.graph['crs'] == districts.crs == buildings.crs:
    raise ValueError('CRS G, districts и buildings различны!')
else:
    print('CRS совпадают.')

Проверка СК: CRS совпадают.


In [5]:
G = ox.project_graph(G)
districts = ox.project_gdf(districts)
buildings = ox.project_gdf(buildings)

print('Проверка СК:', end=' ')
if not G.graph['crs'] == districts.crs == buildings.crs:
    raise ValueError('CRS G, districts и buildings различны!')
else:
    print('CRS совпадают.')

Проверка СК: CRS совпадают.


Дополнение зданий сведениями о ближайших узлах ГДС

In [6]:
g_nodes = ox.graph_to_gdfs(G, edges=False)
buildings = buildings.sjoin_nearest(g_nodes[['geometry']], max_distance=100)
buildings.rename(columns={'index_right': 'node'}, inplace=True)

Разбор графа и зданий на наборы для районов

In [7]:
# g_nodes = ox.graph_to_gdfs(G, edges=False)

scopes = {}
for i, district in districts.iterrows():
    poly = district['geometry']
    print(f'{i}: ', end='')

    # Граф
    g_nodes_in = g_nodes.loc[g_nodes.intersects(poly.buffer(1000))]
    g = nx.subgraph(G, g_nodes_in.index)
    # оставляем только больший сильносвязный компонент:
    # g = max(nx.strongly_connected_components(g), key=len)
    g = nx.subgraph(g, max(nx.strongly_connected_components(g), key=len))
    print(g.number_of_nodes(), g.number_of_edges(), end=' ')

    # Здания
    b = buildings.loc[buildings.intersects(poly)]
    print(len(b))

    scopes[i] = [poly, g, b]

Железнодорожный район: 3515 8411 2898
Кировский район: 5100 12939 3382
Ленинский район: 5953 15022 4812
Октябрьский район: 7008 17201 11798
Свердловский район: 5490 13597 15159
Советский район: 9377 22942 8101
Центральный район: 5641 13763 6440


In [8]:
# ДЛЯ ПЕЧАТИ КАРТЫ РАСКОММЕНТИРОВАТЬ!
# def plot_g(name, g, b, poly, ax):
#     ax.set_title(name)
#     ax.set_axis_off()
#     ax.set_facecolor('white')
#     ox.plot_graph(g, node_size=0, edge_linewidth=0.2, ax=ax, show=False)
#     b.plot(ax=ax)
#     poly.plot(color='none', edgecolor='r', linewidth=1, ax=ax)


# fig, axs = plt.subplots(3,3, tight_layout=True, figsize=(12,12), dpi=300)
# x,y = 0,0
# for district in districts.index:
#     print(f'{district} [{y,x}]: ', end='')
#     g = scopes[district][1]
#     b = scopes[district][2]
#     plot_g(district, g, b, districts.loc[[district]], axs[y,x])
#     x += 1
#     if x >= 3:
#         x = 0
#         y += 1
#     print('ok')

# ax=axs[2,1]
# ax.set_title('г. Красноярск')
# ax.set_axis_off()
# ax.set_facecolor('white')
# districts.plot(column='name', ax=ax, legend=True, legend_kwds={'fontsize': 6, 'markerscale':0.5})

# plt.delaxes(axs[2,2])

# fig.tight_layout()

# Вычислительные эксперименты

## BLP

In [9]:
# Из кэша
tdfexisted = pd.read_csv('blp.csv')
tdfexisted = tdfexisted.set_index('Unnamed: 0')
tdfexisted

,Железнодорожный район время,Кировский район время,Ленинский район время,Октябрьский район время,Свердловский район время,Советский район время,Центральный район время,Железнодорожный район точность,Кировский район точность,Ленинский район точность,...,Свердловский район точность2,Советский район точность2,Центральный район точность2,Железнодорожный район узел,Кировский район узел,Ленинский район узел,Октябрьский район узел,Свердловский район узел,Советский район узел,Центральный район узел
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
Full,351.51,722.88,1405.28,1945.28,1298.06,3470.49,830.0,100.0,100.0,100.0,...,100.0,100.0,100.0,345759063.0,418315034,356754179,315787799,316563423,2007789981.0,290662818.0
HC,5.60,7.60,13.70,15.60,11.80,10.90,7.1,20.3,18.6,4.9,...,82.9,79.3,85.4,345759063.0,944833811,356754179,346095619,416513327,1837295864.0,290664106.0
MSA,83.10,88.40,114.20,159.70,118.90,201.10,104.6,60.0,66.7,46.2,...,95.9,100.0,100.0,346119674.0,356751072,356754179,315787799,316563423,2007789981,290662818.0
HC+BNHD,3.37,11.92,4.94,7.48,12.77,NaN,NaN,95.9,89.6,89.2,...,86.7,NaN,NaN,290662851,944833811,965856797,750722021,600113140,NaN,NaN
MSA+BNHD,146.70,155.20,237.00,307.00,281.30,315.80,198.8,100.0,100.0,100.0,...,100.0,100.0,100.0,345759063,418315034,356754179,315787799,416513311,2007789981,290662818.0
ABC,28.00,38.70,49.20,62.10,51.50,83.00,NaN,92.3,57.9,72.4,...,99.1,100.0,NaN,345759063,418315034,356754179,315787799,416513311,2007789981,NaN
SHS,51.20,86.50,120.50,151.60,130.70,102.30,NaN,71.4,88.9,16.7,...,99.6,99.7,NaN,345759063,418315034,355950683,315787799,416513311,2007789981,NaN
SA,184.03,281.42,306.17,374.67,308.54,1012.36,NaN,98.3,99.5,99.7,...,98.9,100.0,NaN,{9107254441: 'best'},{356383056: 'best'},{355950683: 'best'},{357993142: 'best'},{3186208170: 'best'},{2007789981: 'best'},NaN


In [10]:
# # Из кэша
# tdfexisted = pd.read_csv('blp.csv')
# tdfexisted = tdfexisted.set_index('Unnamed: 0')


def save_experiment_state():
    # Сохранение итогов в основные словари:
    main_times_list[district+' время'] = times_list
    main_accuracy_list[district+' точность'] = accuracy_list
    main_accuracy2_list[district+' точность2'] = accuracy2_list
    main_best_node_list[district+' узел'] = best_node_list
    # ------------------------------
    tdf = pd.DataFrame({**main_times_list, **main_accuracy_list, **main_accuracy2_list, **main_best_node_list})
    tdf.to_csv('blp.csv')


main_times_list = {}
main_accuracy_list = {}
main_accuracy2_list = {}
main_best_node_list = {}

for district, scope in scopes.items():
    print(' ')
    print('='*80)
    print(district)
    g = scopes[district][1]
    b = scopes[district][2]
    nodes = ox.graph_to_gdfs(g, edges=False)
    area_nodes = nodes.within(scopes[district][0])
    nodes_list=set(nodes[area_nodes].index)
    print('Узлов', sum(area_nodes))
    print('='*80)

    # функция метрики спроса
    metric_demand = Demand(b, buildings_demand_field='kfpo_spros')

    # Списки результатов расчета
    times_list = {}
    accuracy_list = {}
    accuracy2_list = {}     # Для метрики по результату
    best_node_list = {}
    t = TimeCheckPoint()
    t2 = TimeCheckPoint()

    # Расчет полным перебором
    # ============================================================================
    print('\n', '# Расчет полным перебором')
    print('-'*80)
    try:
        accuracy_list['Full'] = tdfexisted.loc['Full'][district+' точность']
        accuracy2_list['Full'] = tdfexisted.loc['Full'][district+' точность2']
        times_list['Full'] = tdfexisted.loc['Full'][district+' время']
        best_node_list['Full'] = tdfexisted.loc['Full'][district+' узел']

        # Сохраняем эталонные значения
        etalon_node, etalon_time = best_node_list['Full'], times_list['Full']
        if np.isnan(etalon_time):
            raise ValueError('Значение не было вычислено!')
        etalon_node = int(float(etalon_node))
        etalon_metric = NodeMetric(FirstArrivalUnitState(), metric_demand)(g, etalon_node, area=area_nodes)
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        pb = Progressbar(sum(area_nodes), bins=40)
        full = BestNodesFull(FirstArrivalUnitState(), metric_demand, node_calc_end_function=pb)
        best_nodes, best_metric = full(env=g, area=area_nodes, nodes_list=nodes_list)
        time_spent = t()

        # Сохраняем результат вычисления
        times_list['Full'] = time_spent
        accuracy_list['Full'] = 100.0
        accuracy2_list['Full'] = 100.0
        best_node_list['Full'] = best_nodes[0]
        save_experiment_state()

        # Сохраняем эталонные значения
        etalon_metric = best_metric
        etalon_node, etalon_time = best_node_list['Full'], times_list['Full']
    
    etalon_time = etalon_time if etalon_time < 600 else 600

    print(f'ЭТАЛОННЫЙ УЗЕЛ: {etalon_node}')
    print(f'ЭТАЛОННАЯ МЕТРИКА: {etalon_metric}')
    print(f'РАСЧЕТНОЕ ВРЕМЯ: {etalon_time}')

    # Расчет HC
    # ============================================================================
    alg_name = 'HC'
    print('\n', f'# Расчет {alg_name}')
    print('-'*80)
    try:
        accuracy_list[alg_name] = tdfexisted.loc[alg_name][district+' точность']
        accuracy2_list[alg_name] = tdfexisted.loc[alg_name][district+' точность2']
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        best_node_list[alg_name] = tdfexisted.loc[alg_name][district+' узел']
        if np.isnan(times_list[alg_name]):
            raise ValueError('Значение не было вычислено!')
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)
        bn_list = []
        bn_metrics = []
        i=1
        t()
        while t(False) < etalon_time:
            t2()
            start_node = random.choice(list(nodes_list))
            best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)
            bn_list.append(best_node)
            bn_metrics.append(best_metric)
            print('\n', i, best_node, best_metric, t2(), end='\r')
            i+=1
        time_spent = t()

        # Сохраняем результат вычисления
        correct_results = bn_list.count(etalon_node)
        times_list[alg_name] = round(time_spent/len(bn_list), 1)
        accuracy_list[alg_name] = round(100*correct_results/len(bn_list), 1)
        accuracy2_list[alg_name] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
        best_node_list[alg_name] = best_node
        save_experiment_state()
        print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["HC"]}', '|', accuracy2_list['HC'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["HC"]} сек')

    # Расчет MSA
    # ============================================================================
    alg_name = 'MSA'
    print('\n', f'# Расчет {alg_name}')
    print('-'*80)
    try:
        accuracy_list[alg_name] = tdfexisted.loc[alg_name][district+' точность']
        accuracy2_list[alg_name] = tdfexisted.loc[alg_name][district+' точность2']
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        best_node_list[alg_name] = tdfexisted.loc[alg_name][district+' узел']
        if np.isnan(times_list[alg_name]):
            raise ValueError('Значение не было вычислено!')
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        class EventFunc:
            def __init__(self):
                pass
            def __call__(self, **kwargs):
                global_jump_number = kwargs['global_jump_number']
                local_jump_number = kwargs.get('local_jump_number', 0)
                print("Глобальный прыжок:", global_jump_number, ", Локальный прыжок:", local_jump_number, '  ', end='\r')

        bnmsa = BestNodeMonkey(FirstArrivalUnitState(), metric_demand, 
                                    global_jumps_count=2,
                                    local_jumps_count=5,
                                    all_neighbors=True,
                                    after_global_jump_function=EventFunc(),
                                    after_local_jump_function=EventFunc(),
                                    )
        bn_list = []
        bn_metrics = []
        i=1
        t()
        while t(False) < etalon_time:
            t2()
            best_node, best_metric = bnmsa(env=g, area=area_nodes)
            bn_list.append(best_node)
            bn_metrics.append(best_metric)
            # print('\n', i, best_node, best_metric, t2(), end='\r')
            print(i, best_node, best_metric, t2())
            # print(etalon_metric)
            # print(bn_metrics)
            # print(100 * sum(bn_metrics))
            # print(etalon_metric * len(bn_metrics))
            # print(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)))
            # print(round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1))
            i+=1
        time_spent = t()

        # Сохраняем результат вычисления
        correct_results = bn_list.count(etalon_node)
        times_list['MSA'] = round(time_spent/len(bn_list), 1)
        accuracy_list['MSA'] = round(100*correct_results/len(bn_list),1)
        accuracy2_list['MSA'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
        best_node_list['MSA'] = best_node
        save_experiment_state()
        print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["MSA"]}', '|', accuracy2_list['MSA'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["MSA"]} сек')

    # Расчет HC+BNHD
    # ============================================================================
    alg_name = 'HC+BNHD'
    print('\n', f'# Расчет {alg_name}')
    print('-'*80)
    try:
        accuracy_list[alg_name] = tdfexisted.loc[alg_name][district+' точность']
        accuracy2_list[alg_name] = tdfexisted.loc[alg_name][district+' точность2']
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        best_node_list[alg_name] = tdfexisted.loc[alg_name][district+' узел']
        if np.isnan(times_list[alg_name]):
            raise ValueError('Значение не было вычислено!')
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        bnhd = BestNodesHalfDiameter(FirstArrivalUnitState(), metric_demand)
        bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)

        t()
        start_node = bnhd(env=g, area=area_nodes)
        best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)
        time_spent = t()

        # Сохраняем результат вычисления
        correct_results = best_metric/etalon_metric
        times_list['HC+BNHD'] = time_spent
        accuracy_list['HC+BNHD'] = round(100*correct_results,1)
        accuracy2_list['HC+BNHD'] = round(100*correct_results,1)
        best_node_list['HC+BNHD'] = best_node
        save_experiment_state()
        print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["HC+BNHD"]}', '|', accuracy2_list['HC+BNHD'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["HC+BNHD"]} сек')


    # Расчет MSA+BNHD
    # ============================================================================
    alg_name = 'MSA+BNHD'
    print('\n', f'# Расчет {alg_name}')
    print('-'*80)
    try:
        accuracy_list[alg_name] = tdfexisted.loc[alg_name][district+' точность']
        accuracy2_list[alg_name] = tdfexisted.loc[alg_name][district+' точность2']
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        best_node_list[alg_name] = tdfexisted.loc[alg_name][district+' узел']
        if np.isnan(times_list[alg_name]):
            raise ValueError('Значение не было вычислено!')
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        bnhd = BestNodesHalfDiameter(FirstArrivalUnitState(), metric_demand)
        bnmsa = BestNodeMonkey(FirstArrivalUnitState(), metric_demand, 
                                global_jumps_count=2,
                                local_jumps_count=10,
                                all_neighbors=True,
                                local_jump_max_distance=1500,
                                after_global_jump_function=EventFunc(),
                                after_local_jump_function=EventFunc(),
                                )

        bn_list = []
        bn_metrics = []
        i=1
        t()
        while t(False) < etalon_time:
            t2()
            start_node = bnhd(env=g, area=area_nodes)
            best_node, best_metric = bnmsa(env=g, area=area_nodes, start_node=start_node)
            bn_list.append(best_node)
            bn_metrics.append(best_metric)
            print('\n', i, best_node, best_metric, t2(), end='\r')
            i+=1
        time_spent = t()

        # Сохраняем результат вычисления
        correct_results = bn_list.count(etalon_node)
        times_list['MSA+BNHD'] = round(time_spent/len(bn_list), 1)
        accuracy_list['MSA+BNHD'] = round(100*correct_results/len(bn_list),1)
        accuracy2_list['MSA+BNHD'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
        best_node_list['MSA+BNHD'] = best_node
        save_experiment_state()
        print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["MSA+BNHD"]}', '|', accuracy2_list['MSA+BNHD'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["MSA+BNHD"]} сек')






    # Расчет ABC
    # ============================================================================
    alg_name = 'ABC'
    print('\n', f'# Расчет {alg_name}')
    print('-'*80)
    try:
        accuracy_list[alg_name] = tdfexisted.loc[alg_name][district+' точность']
        accuracy2_list[alg_name] = tdfexisted.loc[alg_name][district+' точность2']
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        best_node_list[alg_name] = tdfexisted.loc[alg_name][district+' узел']
        if np.isnan(times_list[alg_name]):
            raise ValueError('Значение не было вычислено!')
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        bnabc = BestNodeBee(FirstArrivalUnitState(),
                        metric_demand,
                        # scouts_count=150,
                        # best_scouts_count=3,
                        )
        bn_list = []
        bn_metrics = []
        i=1
        t()
        while t(False) < etalon_time:
            t2()
            best_node, best_metric = bnabc(env=g, area=area_nodes, nodes_list=nodes_list)
            bn_list.append(best_node)
            bn_metrics.append(best_metric)
            print('\n', i, best_node, best_metric, t2(), end='\r')
            i+=1
        time_spent = t()

        # Сохраняем результат вычисления
        correct_results = bn_list.count(etalon_node)
        times_list['ABC'] = round(time_spent/len(bn_list), 1)
        accuracy_list['ABC'] = round(100*correct_results/len(bn_list),1)
        accuracy2_list['ABC'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
        best_node_list['ABC'] = best_node
        save_experiment_state()
        print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["ABC"]}', '|', accuracy2_list['ABC'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["ABC"]} сек')

    # Расчет SHS
    # ============================================================================
    alg_name = 'SHS'
    print('\n', f'# Расчет {alg_name}')
    print('-'*80)
    try:
        accuracy_list[alg_name] = tdfexisted.loc[alg_name][district+' точность']
        accuracy2_list[alg_name] = tdfexisted.loc[alg_name][district+' точность2']
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        best_node_list[alg_name] = tdfexisted.loc[alg_name][district+' узел']
        if np.isnan(times_list[alg_name]):
            raise ValueError('Значение не было вычислено!')
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        bnabc = BestNodeBee(FirstArrivalUnitState(),
                        metric_demand,
                        scouts_count=10,
                        best_scouts_count=10,
                        )
        bn_list = []
        bn_metrics = []
        i=1
        t()
        while t(False) < etalon_time:
            t2()
            best_node, best_metric = bnabc(env=g, area=area_nodes, nodes_list=nodes_list)
            bn_list.append(best_node)
            bn_metrics.append(best_metric)
            print('\n', i, best_node, best_metric, t2(), end='\r')
            i+=1
        time_spent = t()

        # Сохраняем результат вычисления
        correct_results = bn_list.count(etalon_node)
        times_list['SHS'] = round(time_spent/len(bn_list), 1)
        accuracy_list['SHS'] = round(100*correct_results/len(bn_list),1)
        accuracy2_list['SHS'] = round(100 * sum(bn_metrics) / (etalon_metric * len(bn_metrics)), 1)
        best_node_list['SHS'] = best_node
        save_experiment_state()
        print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["SHS"]}', '|', accuracy2_list['SHS'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["SHS"]} сек')

    # Расчет SA
    # ============================================================================
    alg_name = 'SA'
    print('\n', f'# Расчет {alg_name}')
    print('-'*80)
    try:
        accuracy_list[alg_name] = tdfexisted.loc[alg_name][district+' точность']
        accuracy2_list[alg_name] = tdfexisted.loc[alg_name][district+' точность2']
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        best_node_list[alg_name] = tdfexisted.loc[alg_name][district+' узел']
        if np.isnan(times_list[alg_name]):
            raise ValueError('Значение не было вычислено!')
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:

        def best_state_find_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
            print(f'\nШаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |')
        def turn_end_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
            print(f'Шаг: {turn} |', f'T={round(T,5)} |', 
                  f'Лучшая метрика: {best_metric} |',
                  f'Текущая метрика: {cur_metric} |',
                  f'Лучшее положение: {best_bot} |',
                  end='\r')        

        node_selector = RandomNodesSelector(nodes_list=nodes_list)
        stop_case = NSameStopCase(n=2000)
        mclpsa = BestNodesSA(FirstArrivalUnitState(),
                            metric_demand,
                            node_selector=node_selector,
                            end_temperature=0.0001,
                            turns= 10000,
                             turn_end_function=turn_end_function,
                            # best_state_find_function=best_state_find_function,
                            stop_case_function=stop_case
                            )

        t()
        # best_node, best_metric = mclpsa(env=g, area=area_nodes, start_node=start_node)
        print({list(g.nodes())[0]:'best'})
        best_node, best_metric = mclpsa(env=g, area=area_nodes, dynamic_nodes={list(g.nodes())[0]:'best'})
        time_spent = t()

        # Сохраняем результат вычисления
        correct_results = best_metric/etalon_metric
        times_list['SA'] = time_spent
        accuracy_list['SA'] = round(100*correct_results,1)
        accuracy2_list['SA'] = round(100*correct_results,1)
        best_node_list['SA'] = best_node
        save_experiment_state()
        print('\n')
    print(f'ТОЧНОСТЬ: {accuracy_list["SA"]}', '|', accuracy2_list['SA'])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list["SA"]} сек')

    # ======================================================================
    # Сохранение итогов в основные словари:
    save_experiment_state()

    # До этого:
    # main_times_list[district+' время'] = times_list
    # main_accuracy_list[district+' точность'] = accuracy_list
    # main_accuracy2_list[district+' точность2'] = accuracy2_list
    # main_best_node_list[district+' узел'] = best_node_list
    # # Промежуточное сохранение в DataFrame с объединением main_times_list и main_accuracy_list
    # # ------------------------------
    # # tdf_times = pd.DataFrame(main_times_list)
    # # tdf_times.to_csv('times.csv')
    # # tdf_accuracy = pd.DataFrame(main_accuracy_list)
    # # tdf_accuracy.to_csv('accuracy.csv')
    # tdf = pd.DataFrame({**main_times_list, **main_accuracy_list, **main_accuracy2_list, **main_best_node_list})
    # tdf.to_csv('metrics.csv')
    # # tdf.to_csv(f'{district}.csv')

    

 
Железнодорожный район
Узлов 1861

 # Расчет полным перебором
--------------------------------------------------------------------------------
ИМЕЕТСЯ В КЭШЕ!
ЭТАЛОННЫЙ УЗЕЛ: 345759063
ЭТАЛОННАЯ МЕТРИКА: 0.00111746411505084
РАСЧЕТНОЕ ВРЕМЯ: 351.51

 # Расчет HC
--------------------------------------------------------------------------------
ИМЕЕТСЯ В КЭШЕ!
ТОЧНОСТЬ: 20.3 | 91.2
ВРЕМЯ ОДНОГО РАСЧЕТА: 5.6 сек

 # Расчет MSA
--------------------------------------------------------------------------------
ИМЕЕТСЯ В КЭШЕ!
ТОЧНОСТЬ: 60.0 | 98.2
ВРЕМЯ ОДНОГО РАСЧЕТА: 83.1 сек

 # Расчет HC+BNHD
--------------------------------------------------------------------------------
ИМЕЕТСЯ В КЭШЕ!
ТОЧНОСТЬ: 95.9 | 95.9
ВРЕМЯ ОДНОГО РАСЧЕТА: 3.37 сек

 # Расчет MSA+BNHD
--------------------------------------------------------------------------------
ИМЕЕТСЯ В КЭШЕ!
ТОЧНОСТЬ: 100.0 | 100.0
ВРЕМЯ ОДНОГО РАСЧЕТА: 146.7 сек

 # Расчет ABC
------------------------------------------------------------------

In [ ]:
# tdf

In [ ]:
# # Сохранение итогов в основные словари:
# from tabulate import tabulate

# print(
#     tabulate(tdf,
#              headers='keys',
#              tablefmt='pipe'),
#     )

# MCLP

In [ ]:
# Из кэша
tdfexisted = pd.read_csv('mclp.csv')
tdfexisted = tdfexisted.set_index('Unnamed: 0')
# tdfexisted = tdfexisted.drop('GA+GNS')
tdfexisted

,Железнодорожный район время,Кировский район время,Железнодорожный район точность,Кировский район точность
Unnamed: 0,,,,
SA,624.04,668.41,0.001474,0.001625
GA,277.77,441.89,0.001529,0.001700
GNS,12.60,10.44,0.001453,0.001437
GA+GNS,288.30,423.75,0.001588,0.001713


In [ ]:

main_times_list = {}
main_metrics_list = {}

units_count = 3

for district, scope in scopes.items():
    print(' ')
    print('='*80)
    print(district)
    g = scopes[district][1]
    b = scopes[district][2]
    nodes = ox.graph_to_gdfs(g, edges=False)
    area_nodes = nodes.within(scopes[district][0])
    nodes_list=set(nodes[area_nodes].index)
    start_state = {random.sample(list(nodes_list), k=1)[0]:f'псч {i}' for i in range(units_count)}
    print('Узлов', sum(area_nodes))
    print('='*80)

    # функция метрики спроса
    metric_demand = Demand(b, buildings_demand_field='kfpo_spros')

    # Списки результатов расчета
    times_list = {}
    metrics_list = {}

    # Расчет SA
    # ============================================================================
    print('\n', '# SA')
    print('-'*80)

    try:
        best_metric = tdfexisted.loc['SA'][district+' точность']
        time_spent = tdfexisted.loc['SA'][district+' время']
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        t = TimeCheckPoint()
        t2 = TimeCheckPoint()

        # здесь сам расчет
        def turn_end_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
            print(f'Шаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |', ' '*10, end='\r')
        def best_state_find_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
            print(f'\nШаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |')
        point_selector = RandomNodesSelector(nodes_list=nodes_list)
        stop_case = NSameStopCase(n=2000)
        mclpsa = BestNodesSA(FirstArrivalUnitState(),
                            metric_demand,
                            node_selector = point_selector,
                            end_temperature = 0.0001,
                            turns = 10000,
                            mutation_max_count = 2,
                            turn_end_function=turn_end_function,
                            # best_state_find_function = best_state_find_function,
                            stop_case_function = stop_case
                            )
        best_state, best_metric = mclpsa(env=g, area=area_nodes, dynamic_nodes=start_state)

        # завершающая стадия расчета
        time_spent = t()
        print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['SA'] = time_spent
    metrics_list['SA'] = best_metric


    # Расчет GA
    # ============================================================================
    print('\n', '# GA')
    print('-'*80)
    try:
        best_metric = tdfexisted.loc['GA'][district+' точность']
        time_spent = tdfexisted.loc['GA'][district+' время']
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        t = TimeCheckPoint()
        # t2 = TimeCheckPoint()

        # здесь сам расчет
        def epoch_end_function(epoch, best_metric, cur_metric, **kwargs):
            print(f'Эпоха: {epoch} |', f'Лучшая метрика: {best_metric} |', f'Текущая метрика: {cur_metric} |', ' '*10, end='\r')
        point_selector = RandomNodesSelector(nodes_list=nodes_list)
        stop_case = NSameStopCase(n=20)
        mclpga = BestNodesGA(FirstArrivalUnitState(),
                        metric_demand,
                        node_selector = point_selector,
                        population_size=50,
                        elite_size=15,
                        epochs=30,
                        mutation_rate=0.5,
                        mutation_max_count=3,
                        epoch_end_function=epoch_end_function,
                        # stop_case_function = stop_case
                        )
        optimal_nodes, best_metric = mclpga(env=g, 
                                    dynamic_nodes=start_state,
                                    area=area_nodes,
                                    )

        # завершающая стадия расчета
        time_spent = t()
        print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['GA'] = time_spent
    metrics_list['GA'] = best_metric


    # Расчет GNS
    # ============================================================================
    print('\n', '# GNS')
    print('-'*80)
    try:
        best_metric = tdfexisted.loc['GNS'][district+' точность']
        time_spent = tdfexisted.loc['GNS'][district+' время']
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        t = TimeCheckPoint()
        # t2 = TimeCheckPoint()

        # здесь сам расчет
        def iter_func(i, best_metric, **kwds):
            print(i, ':', best_metric, end='\r')
        bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
                        metric_function=metric_demand,
                        )
        mclpgns = BestNodesKoptG(state_function=FirstArrivalUnitState(),
                        metric_function=metric_demand,
                        best_point_function=bnhc,
                        iterations=5,
                        iter_calc_end_function=iter_func,
                        )
        optimal_nodes, best_metric = mclpgns(env=g,
                            dynamic_nodes=start_state,
                            area=area_nodes,
                            )

        # завершающая стадия расчета
        time_spent = t()
        print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['GNS'] = time_spent
    metrics_list['GNS'] = best_metric


    # Расчет GNS+BNHD
    # ============================================================================
    alg_name = 'GNS+BNHD'
    print('\n', f'# {alg_name}')
    print('-'*80)
    try:
        best_metric = tdfexisted.loc[alg_name][district+' точность']
        time_spent = tdfexisted.loc[alg_name][district+' время']
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        t = TimeCheckPoint()
        # t2 = TimeCheckPoint()

        # здесь сам расчет
        def iter_func(i, best_metric, **kwds):
            print(i, ':', best_metric, end='\r')
        bnhc = BestNodeHillClimbingHD(state_function=FirstArrivalUnitState(),
                        metric_function=metric_demand,
                        )
        mclpgns = BestNodesKoptG(state_function=FirstArrivalUnitState(),
                        metric_function=metric_demand,
                        best_point_function=bnhc,
                        iterations=5,
                        iter_calc_end_function=iter_func,
                        )
        optimal_nodes, best_metric = mclpgns(env=g,
                            dynamic_nodes=start_state,
                            area=area_nodes,
                            )

        # завершающая стадия расчета
        time_spent = t()
        print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list[alg_name] = time_spent
    metrics_list[alg_name] = best_metric


    # Расчет GA+GNS
    # ============================================================================
    print('\n', '# GA+GNS')
    print('-'*80)
    try:
        best_metric = tdfexisted.loc['GA+GNS'][district+' точность']
        time_spent = tdfexisted.loc['GA+GNS'][district+' время']
        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        t = TimeCheckPoint()
        # t2 = TimeCheckPoint()

        # здесь сам расчет
        def epoch_end_function(epoch, best_metric, cur_metric, **kwargs):
            print(f'Эпоха: {epoch} |', f'Лучшая метрика: {best_metric} |', f'Текущая метрика: {cur_metric} |', ' '*10, end='\r')
        def iter_calc_end_function(i, best_metric, **kwargs):
            print(f'Итерация: {i} |', f'Лучшая метрика: {best_metric} |', ' '*10)  #, end='\r')
        # optimal_nodes, best_metric = mclpga(env=g, 
        #                               dynamic_nodes=start_state,
        #                               area=area_nodes,
        #                               )
        # optimal_nodes, best_metric = mclpgns(env=g,
        #                     dynamic_nodes=optimal_nodes,
        #                     area=area_nodes,
        #                     )
        bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
                    metric_function=metric_demand,                      
                        )
        kopt = BestNodesKoptG(FirstArrivalUnitState(),
                            metric_function=metric_demand,
                            best_point_function=bnhc,
                            iterations=3,
                            iter_calc_end_function=iter_calc_end_function,
                            )
        point_selector = RandomNodesSelector(nodes_list=nodes_list)
        stop_case = NSameStopCase(n=20)
        mclp = timing(BestNodesGAKopt(FirstArrivalUnitState(),
                            metric_function=metric_demand,
                            kopt_function = kopt,
                            node_selector = point_selector,
                            population_size=50,
                            elite_size = 15,
                            epochs = 30,
                            mutation_rate = 0.5,
                            mutation_max_count=3,
                            epoch_end_function = epoch_end_function,
                            #    stop_case_function=stop_case,
        ))

        # Собственно применения построенного алгоритма
        optimal_nodes, best_metric = mclp(env=g,
                                    #   static_nodes=existed_units, 
                                    dynamic_nodes=start_state,
                                    area=area_nodes,
                                    )

        # завершающая стадия расчета
        time_spent = t()
        print('\n')
    print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

    # Сохраняем результат вычисления
    times_list['GA+GNS'] = time_spent
    metrics_list['GA+GNS'] = best_metric




    # Сохранение итогов в основные словари:
    main_times_list[district+' время'] = times_list
    main_metrics_list[district+' точность'] = metrics_list
    
    # Промежуточное сохранение в DataFrame с объединением main_times_list и main_metrics_list
    # ------------------------------
    tdf = pd.DataFrame({**main_times_list, **main_metrics_list})
    tdf.to_csv('mclp.csv')
    # tdf.to_csv(f'mclp {district}.csv')



In [24]:
tdfexisted = pd.read_csv('mclp.csv')
tdfexisted = tdfexisted.set_index('Unnamed: 0')
tdfexisted = tdfexisted.drop('GA+GNS')
tdfexisted

,Железнодорожный район время,Кировский район время,Железнодорожный район точность,Кировский район точность
Unnamed: 0,,,,
SA,624.04,668.41,0.001474,0.001625
GA,277.77,441.89,0.001529,0.001700
GNS,12.60,10.44,0.001453,0.001437


# LSCP

In [ ]:
# Из кэша
tdfexisted = pd.read_csv('lscp.csv')
tdfexisted = tdfexisted.set_index('Unnamed: 0')
tdfexisted

In [39]:
# from genesis.core import MCLPBase, PointSelectorBase, StateBase
# from genesis.tools import list_dict_concat


# class BestNodesGADiss(MCLPBase):
#     '''
#     Поиск лучших узлов для размещения n объектов.

#     Расчет производится генетическим алгоритмом.

#     ## Область применения
#     Определение размещения множества узлов в графе, при котором 
#     обеспечиваются наилучшие некоторые целевые метрики. 
    
#     Дает решение высокой точности. Хорошо подходит для больших графов.
#     '''

#     def __init__(self,
#                  state_function: StateBase,
#                  metric_function: MetricBase,
#                  node_selector:PointSelectorBase,
#                  population_size:int = 25,
#                  epochs:int = 50,
#                  mutation_rate:float = 0.5,
#                  elite_size:int = 0,
#                  appr_val_in_area:float = 0,
#                  mutation_max_count:int = 1,
#                  bad_val_in_area:int = 1000,
#                  epoch_end_function:callable = None,
#                  stop_case_function:callable = None,
#                  rise_rate:float = 0.1,
#                  decrease_rate:float = 0.1,
#                  names_pattern: str = '{}',
#                  start_names_index: int = 1,
#                  **kwargs) -> None:
#         '''
#         ## Аргументы
#         `state_function`: StateBase
#             функция расчета состояния окружения
#         `metric_function`: MetricBase
#             Функция расчета метрики
#         `node_selector`: PointSelectorBase
#             Функция выбора узла.
#         `population_size`:int = 25
#             Размер популяции
#         `epochs`: int = 50
#             Количество эпох расчета
#         `mutation_rate`: float = 0.1
#             Вероятность единичной мутации
#         `elite_size`: int =  0
#             Количество особей в элитной группе.
#             Если указан 0, то механизм элитизма не задействуется.
#         `appr_val_in_area`: int=0
#             Доля узлов графа в пределах area, покрытие которой считается приемлемой для принятия расчетной метрики. 
#             Если при расчете метрик, из стартового узла (узлов) достижимо меньшее количество узлов,
#             то такой узел не рассматривается.
#             При расчет размещения нескольких узлов неминуемо возникает ситуация при которой
#             часть территории area будет недостижима. Поэтому по умолчанию считается
#         `mutation_max_count`: int = 1
#             Максимальное количество единиц мутации.
#         `bad_val_in_area`: int=1000
#             Значение указываемое для узла, в случае если
#             из него нельзя попасть в `appr_val_in_area` долю узлов в пределах
#             `area`.
#         `epoch_end_function`: callable = None
#             Функция вызываемая после каждой эпохи расчета.
#         `stop_case_function`: callable = None
#             Функция проверки достигнута ли цель расчета
#         '''
#         if population_size <= 5:
#             raise ValueError('Размер популяции должен быть > 0, так же не рекомендуется использовать размер популяции < 5')
#         if epochs  <=  0:
#             raise ValueError('Количество эпох должно быть > 0')
#         if mutation_max_count  <=  0:
#             raise ValueError('Максимальное количество единиц мутации должно быть > 0')
#         if mutation_rate <0 or mutation_rate > 1:
#             raise ValueError('Вероятность мутации должна лежать в диапазоне (0,1)')
#         if appr_val_in_area <0 or appr_val_in_area > 1:
#             raise ValueError('Доля узлов графа в пределах area, должна лежать в диапазоне (0,1)')
#         if elite_size < 0:
#             raise ValueError('Количество особей в элитной группе должно быть >= 0')
        
#         self.population_size = population_size
#         self.epochs = epochs
#         self.mutation_rate = mutation_rate
#         self.elite_size = elite_size
#         self.appr_val_in_area = appr_val_in_area
#         self.mutation_max_count = mutation_max_count
#         self.bad_val_in_area = bad_val_in_area
#         self.epoch_end_function = epoch_end_function
#         self.stop_case_function = stop_case_function
#         self.node_selector = node_selector
#         self.rise_rate = rise_rate
#         self.decrease_rate = decrease_rate
#         self.names_pattern = names_pattern
#         self.names_index = start_names_index
#         super().__init__(state_function, metric_function, **kwargs)

#     def _fit_function(self, env, nodes, area=None, **kwargs):
#         '''
#         Расчет функции приспособленности
#         '''
#         # 2.1. Расчет состояния
#         times, _ = self.state_function(env=env, points=nodes, area=area, **kwargs)

#         if not area is None:
#             appr_nodes_count = area.sum() * self.appr_val_in_area
#             if len(times) < appr_nodes_count:
#                 print('Расстановка не обеспечивает требуемую степень прикрытия территории area')
#                 return self.bad_val_in_area

#         # 2.2. Расчет стартовой метрики состояния и определение стартового размещения
#         best_metric = self.metric_function(times, **kwargs)

#         return best_metric / 0.5*len(nodes)
#         # return len(nodes) / best_metric
#         # return best_metric

#     def _mutate(self, env, new_dynamic_nodes, area):
#         '''
#         Мутация особи
#         '''
#         for _ in range(self.mutation_max_count):
#             if random.random() < self.mutation_rate:
#                 # Выбор случайного элемента в словаре и удаление его из new_dynamic_nodes
#                 node, unit = random.choice(list(new_dynamic_nodes.items()))
#                 del new_dynamic_nodes[node]

#                 # Поиск нового узла, которого при этом нет в new_dynamic_nodes
#                 node = self.node_selector(env=env, points=new_dynamic_nodes, area=area)

#                 new_dynamic_nodes[node] = unit

#         # Добавление или удаление новых узлов (расширение или сужение генома)
#         if random.random() < self.decrease_rate and len(new_dynamic_nodes)>2:
#             node, unit = random.choice(list(new_dynamic_nodes.items()))
#             del new_dynamic_nodes[node]
#         if random.random() < self.rise_rate:
#             node = self.node_selector(env=env, points=new_dynamic_nodes, area=area)
#             new_dynamic_nodes[node] = self.names_pattern.format(self.names_index)
#             self.names_index += 1
#             # print(new_dynamic_nodes)

#         return  new_dynamic_nodes

#     def __call__(self,
#                  env:nx.MultiDiGraph,
#                  dynamic_nodes: dict,
#                  static_nodes: dict = None,
#                  area: pd.Series = None,
#                  **kwargs):
#         '''
#         Запуск работы генетического алгоритма

#         ## Аргументы
#         `env`:nx.MultiDiGraph
#             Граф улично-дорожной сети
#         `dynamic_nodes`: list|dict
#             Список стартовых узлов графа в которых размещены
#             подразделения оптимальные места которых следует определить.
#         `static_nodes`: list|dict = None
#             Список стартовых узлов графа в которых размещены
#             подразделения изменять размещение которых не следует.
#         `area`: pd.Series = None
#             Маска узлов графа. Значениями True отмечены узлы графа - цели расчета леса Вороного.
#             Если не указана, расчет производится для всех узлов графа.
#         '''

#         # 0. Проверка корректности пришедших данных
#         if not isinstance(env, nx.MultiDiGraph):
#             raise TypeError("Тип аргумента `env` должен быть `MultiDiGraph`!")
#         if not isinstance(dynamic_nodes, dict):
#             raise TypeError("Тип аргумента `dynamic_nodes` должен быть `dict`!")
#         if not static_nodes is None and not isinstance(static_nodes, dict):
#             raise TypeError("Тип аргумента `static_nodes` должен быть `dict`!")
#         if len(dynamic_nodes) < 2:
#             raise ValueError(f'Количество элементов `dynamic_nodes` не может быть меньше 2. Сейчас {len(dynamic_nodes)}')
#         if not area is None and not isinstance(area, pd.Series):
#             raise TypeError(f'Аргумент `area` должен иметь тип `pd.Series`! Имеет {type(area)}')


#         # 1.1. Если статические узлы не указаны - заменяем значение переменной с None
#         # на {}
#         if static_nodes is None:
#             static_nodes = {}

#         # Последовательность узлов графа (для последующего обращения к нему)
#         g_nodes = list(env.nodes())

#         # ===================================== Генетический алгоритм ==============
#         # 1. Создание стартовой популяции
#         population = [dynamic_nodes for _ in range(self.population_size)]
#         # Оценка приспособленности всех особей
#         bot_fit = [self._fit_function(env=env,
#                                       nodes=list_dict_concat(dn, static_nodes),
#                                       area=area,
#                                       **kwargs) for dn in population]
#         if self.metric_function.compare(1,2)==1:        # Для минимизации:
#             max_val  = max(bot_fit)
#             weights = [1.1*max_val-x for x in bot_fit]
#             best_metric = min(bot_fit)
#         else:                                           # Для максимизации:
#             weights = bot_fit
#             best_metric = max(bot_fit)
#         best_bot = population[bot_fit.index(best_metric)]

#         # 2. На каждой эпохе
#         for epoch in range(self.epochs):

#             # 2.0 Элитарность
#             if self.elite_size>0:
#                 # Определение весов
#                 pop_weight = pd.DataFrame({'w': weights, 'p': population})
#                 pop_weight = pop_weight.sort_values('w', ascending=False)
#                 pop_weight = pop_weight.iloc[:self.elite_size]
#                 population = pop_weight['p'].to_list()
#                 weights = pop_weight['w'].to_list()
                
#                 new_population = population.copy()
#             else:
#                 new_population = []

#             # 2.1 Генерация новой популяции
#             for _ in range(self.population_size - self.elite_size):
#                 # print(len(population), len(weights))

#                 # Отбор по правилу рулетки
#                 parent_bot_1 = random.choices(population, weights=weights)[0]
#                 parent_bot_2 = random.choices(population, weights=weights)[0]

#                 # Скрещивание (одноточечное)
#                 split_point = int(len(parent_bot_1)/2)
#                 left_gen_vals = list(parent_bot_1.values())[:split_point]
#                 right_gen_vals  = list(parent_bot_1.values())[split_point:]
#                 left_genome_part = {k:v for k, v in parent_bot_1.items() if v in left_gen_vals}
#                 right_genome_part = {k:v for k, v in parent_bot_2.items() if v in right_gen_vals}
#                 new_dynamic_nodes = {**left_genome_part, **right_genome_part}

#                 # Мутация (выбор произвольного узла)
#                 new_dynamic_nodes = self._mutate(env=env,
#                                                  new_dynamic_nodes=new_dynamic_nodes,
#                                                  area=area)

#                 # Добавляем его в новую популяцию
#                 new_population.append(new_dynamic_nodes)

#             # 2.2 Заменяем предыдущую популяцию новой
#             population = new_population

#             # 2.3 Оценка приспособленности всех особей
#             bot_fit = [self._fit_function(env=env,
#                                       nodes=list_dict_concat(dn, static_nodes),
#                                       area=area,
#                                       **kwargs) for dn in population]

#             # 2.4 Определение лучшего на эпохе значения метрики
#             # и весов в зависимости от приспособленности        
#             if self.metric_function.compare(1,2)==1:    # Для минимизации:
#                 max_val  = max(bot_fit)
#                 weights = [max_val-x for x in bot_fit]
#                 cur_metric = min(bot_fit)
#             else:                                       # Для максимизации
#                 weights = bot_fit
#                 cur_metric = max(bot_fit)
            
#             # 2.5 Нормализация весов (для более выраженной точности расчета)
#             weights = (weights - np.min(weights)) / (np.max(weights) - np.min(weights))

#             # 2.6 Определяем лучше ли лучшее на эпохе решение чем имеющееся
#             if self.metric_function.compare(best_metric, cur_metric) == cur_metric:
#                 best_bot = population[bot_fit.index(cur_metric)]
#                 best_metric  = cur_metric

#             # 2.7 Выполнение функции окончания расчета на эпохе
#             if self.epoch_end_function:
#                 self.epoch_end_function(epoch=epoch,
#                                         best_metric=best_metric,
#                                         cur_metric=cur_metric,
#                                         best_bot=best_bot)

#             # 2.8 Если достигнута цель расчета, выходим из цикла
#             if self.stop_case_function:
#                 if self.stop_case_function(value=best_metric,
#                             iteration=epoch,
#                             best_metric=best_metric,
#                             dynamic_nodes=best_bot,
#                             static_nodes=static_nodes):
#                     break

#         return best_bot, best_metric

In [ ]:
# # Из кэша
# tdfexisted = pd.read_csv('lscp.csv')
# tdfexisted = tdfexisted.set_index('Unnamed: 0')

from diss.lscp import BestNodesGADiss, BestNodesSADiss
from genesis.lscp import LSCPCommon


runs_count = 1
units_count = 2
target_ip = 90
ip_val_main = 5


def save_experiment_state_lscp():
    # Сохранение итогов в основные словари:
    main_times_list[district+' время'] = times_list
    main_metric_list[district+' метрика'] = metric_list
    main_units_count_list[district+' подразделений'] = units_count_list
    # ------------------------------
    tdf = pd.DataFrame({**main_times_list, **main_metric_list, **main_units_count_list})
    tdf.to_csv('lscp.csv')

# STOPCASE Здесь!
def stop_case_function(value,
                        iteration,
                        best_metric,
                        dynamic_nodes,
                        static_nodes,
                        target_ip=target_ip,
                        ip_val=ip_val_main):
    times, _ = FirstArrivalUnitState()(env=g, points=dynamic_nodes, area=area_nodes)
    ip = CoverIndexBuilding(buildings=b, ip_val=ip_val)(state=times, area=area_nodes)
    # print('\n', ip)
    if ip>= target_ip:
        return True
    return False

main_times_list = {}            # Время одной итерации
main_metric_list = {}           # Лучшая метрика полученная алгоритмом
main_units_count_list = {}      # Количество подразделений предложенное алгоритмом

for district, scope in scopes.items():
    print(' ')
    print('='*80)
    print(district)
    g = scopes[district][1]
    b = scopes[district][2]
    nodes = ox.graph_to_gdfs(g, edges=False)
    area_nodes = nodes.within(scopes[district][0])
    nodes_list=set(nodes[area_nodes].index)
    print('Узлов', sum(area_nodes))
    print('='*80)

    # функция метрики спроса
    metric_demand = Demand(b, buildings_demand_field='kfpo_spros')  #! учет влияния кол-ва подразделений зашит внутри GA и SA

    # Списки результатов расчета
    times_list = {}             # Время одной итерации
    metric_list = {}            # Лучшая метрика полученная алгоритмом
    units_count_list = {}       # Количество подразделений предложенное алгоритмом
    t = TimeCheckPoint()
    t2 = TimeCheckPoint()

    # # Расчет GA
    # # ============================================================================
    # alg_name = 'GA'
    # print('\n', '# Расчет {}'.format(alg_name))
    # print('-'*80)
    # try:
    #     times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
    #     metric_list[alg_name] = tdfexisted.loc[alg_name][district+' метрика']
    #     units_count_list[alg_name] = tdfexisted.loc[alg_name][district+' подразделений']

    #     print(f'ИМЕЕТСЯ В КЭШЕ!')
    # except:
    #     # здесь сам расчет
    #     def epoch_end_function(epoch, best_metric, cur_metric, best_bot, **kwargs):
    #         print(f'Эпоха: {epoch} |', f'Лучшая метрика: {best_metric} |', f'Текущая метрика: {cur_metric} |', best_bot, ' '*10, end='\r')
    #     point_selector = RandomNodesSelector(nodes_list=nodes_list)
    #     metric_build = CoverIndexBuilding(b, ip_val=ip_val_main)
    #     mclpga = BestNodesGADiss(FirstArrivalUnitState(),
    #                     # metric_demand,
    #                     metric_build,
    #                     node_selector = point_selector,
    #                     population_size=50,
    #                     elite_size=15,
    #                     epochs=30,
    #                     mutation_rate=0.5,
    #                     mutation_max_count=3,
    #                     epoch_end_function=epoch_end_function,
    #                     stop_case_function = stop_case_function
    #                     )

    #     bn_times = []
    #     bn_metrics = []
    #     bn_units = []
    #     t()
    #     for i in range(runs_count):
    #         t2()
    #         start_state = {random.sample(list(nodes_list), k=1)[0]:f'псч {i}' for i in range(units_count)}
    #         optimal_nodes, best_metric = mclpga(env=g, 
    #                                 dynamic_nodes=start_state,
    #                                 area=area_nodes,
    #                                 )
    #         bn_times.append(t2(False))
    #         bn_metrics.append(best_metric)
    #         bn_units.append(len(optimal_nodes))
    #         print('\n', i+1, 'Подразделений:', len(optimal_nodes), best_metric, t2(), end='\r')
    #     time_spent = t()

    #     # Сохраняем результат вычисления
    #     times_list[alg_name] = np.mean(bn_times)
    #     metric_list[alg_name] = np.max(bn_metrics)
    #     units_count_list[alg_name] = np.min(bn_units)

    #     save_experiment_state_lscp()
    #     print('\n')
    # print(f'МЕТРИКА: {metric_list[alg_name]}', '| ПОДРАЗДЕЛЕНИЙ', units_count_list[alg_name])
    # print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list[alg_name]} сек')


    # # Расчет SA
    # # ============================================================================
    # alg_name = 'SA'
    # print('\n', '# Расчет {}'.format(alg_name))
    # print('-'*80)
    # try:
    #     times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
    #     metric_list[alg_name] = tdfexisted.loc[alg_name][district+' метрика']
    #     units_count_list[alg_name] = tdfexisted.loc[alg_name][district+' подразделений']

    #     print(f'ИМЕЕТСЯ В КЭШЕ!')
    # except:
    #     # здесь сам расчет
    #     def turn_end_function(turn:int, T:float, best_metric:float, cur_metric:float, best_bot:dict):
    #         print(f'Шаг: {turn} |', f'T={T} |', f'Лучшая метрика: {best_metric} |', f'Лучшее положение: {best_bot} |', ' '*10, end='\r')
    #     point_selector = RandomNodesSelector(nodes_list=nodes_list)
    #     mclpsa = BestNodesSADiss(FirstArrivalUnitState(),
    #                         metric_demand,
    #                         node_selector = point_selector,
    #                         end_temperature = 0.0001,
    #                         turns = 10000,
    #                         mutation_max_count = 2,
    #                         turn_end_function=turn_end_function,
    #                         # best_state_find_function = best_state_find_function,
    #                         stop_case_function = stop_case_function
    #                         )

    #     bn_times = []
    #     bn_metrics = []
    #     bn_units = []
    #     t()
    #     for i in range(runs_count):
    #         t2()
    #         optimal_nodes, best_metric = mclpsa(env=g, 
    #                                 dynamic_nodes=start_state,
    #                                 area=area_nodes,
    #                                 )
    #         bn_times.append(t2(False))
    #         bn_metrics.append(best_metric)
    #         bn_units.append(len(optimal_nodes))
    #         print('\n', i+1, 'Подразделений:', len(optimal_nodes), best_metric, t2(), end='\r')
    #     time_spent = t()

    #     # Сохраняем результат вычисления
    #     times_list[alg_name] = np.mean(bn_times)
    #     metric_list[alg_name] = np.max(bn_metrics)
    #     units_count_list[alg_name] = np.min(bn_units)

    #     save_experiment_state_lscp()
    #     print('\n')
    # print(f'МЕТРИКА: {metric_list[alg_name]}', '| ПОДРАЗДЕЛЕНИЙ', units_count_list[alg_name])
    # print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list[alg_name]} сек')


    # Расчет GNS
    # ============================================================================
    alg_name = 'GNS'
    print('\n', '# Расчет {}'.format(alg_name))
    print('-'*80)
    try:
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        metric_list[alg_name] = tdfexisted.loc[alg_name][district+' метрика']
        units_count_list[alg_name] = tdfexisted.loc[alg_name][district+' подразделений']

        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        # здесь сам расчет
        def iter_func(i, best_metric, **kwds):
            print(i, best_metric, end='\r')
        def mclp_end_function(iteration, best_metric, dynamic_nodes, **kwargs):
            print(f'\nИтерация: {iteration} |', f'Лучшая метрика: {best_metric} |', f'Подразделений: {len(dynamic_nodes)}')
        metric_ipbuild = CoverIndexBuilding(b, ip_val=ip_val_main)
        stop_case = MoreEqualStopCase(target_ip)
        bnhc = BestNodeHillClimbingHD(state_function=FirstArrivalUnitState(),
                        metric_function=metric_demand,
                        )
        mclpgns = BestNodesKoptG(state_function=FirstArrivalUnitState(),
                        metric_function=metric_ipbuild,
                        best_point_function=bnhc,
                        iterations=5,
                        iter_calc_end_function=iter_func,
                        stop_case_function = stop_case_gns
                        )
        point_selector = RandomNodesSelector(nodes_list=nodes_list)
        lscp = LSCPCommon(mclp_function = mclpgns,
                     point_selector = point_selector,
                    stop_case_function = stop_case,
                    start_names_index = units_count+1,
                    names_pattern='псч {}',
                    after_mclp_function=mclp_end_function,
                    )
        start_state = {random.sample(list(nodes_list), k=1)[0]:f'псч {i+1}' for i in range(units_count)}
        

        bn_times = []
        bn_metrics = []
        bn_units = []
        t()
        for i in range(runs_count):
            t2()
            optimal_nodes, best_metric = lscp(env=g,
                            dynamic_nodes=start_state,
                            area=area_nodes,
                            )
            bn_times.append(t2(False))
            bn_metrics.append(best_metric)
            bn_units.append(len(optimal_nodes))
            print('\n', 'Расчет LSCP:')
            print(i+1, 'Подразделений:', len(optimal_nodes), best_metric, t2(), end='\r')
        time_spent = t()

        # Сохраняем результат вычисления
        times_list[alg_name] = np.mean(bn_times)
        metric_list[alg_name] = np.max(bn_metrics)
        units_count_list[alg_name] = np.min(bn_units)

        save_experiment_state_lscp()
        print('\n')
    print(f'МЕТРИКА: {metric_list[alg_name]}', '| ПОДРАЗДЕЛЕНИЙ', units_count_list[alg_name])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list[alg_name]} сек')


    # Расчет GA+GNS
    # ============================================================================
    alg_name = 'GA+GNS'
    print('\n', '# Расчет {}'.format(alg_name))
    print('-'*80)
    try:
        times_list[alg_name] = tdfexisted.loc[alg_name][district+' время']
        metric_list[alg_name] = tdfexisted.loc[alg_name][district+' метрика']
        units_count_list[alg_name] = tdfexisted.loc[alg_name][district+' подразделений']

        print(f'ИМЕЕТСЯ В КЭШЕ!')
    except:
        # здесь сам расчет
        def iter_func(i, best_metric, **kwds):
            print(i, best_metric,kwds, end='\r')
        def epoch_end_function(epoch, best_metric, cur_metric, **kwargs):
            print(f'Эпоха: {epoch} |', f'Лучшая метрика: {best_metric} |', f'Текущая метрика: {cur_metric} |', ' '*10, end='\r')
        

        bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
                    metric_function=metric_demand,                      
                        )
        kopt = BestNodesKoptG(FirstArrivalUnitState(),
                            metric_function=metric_demand,
                            best_point_function=bnhc,
                            iterations=3,
                            iter_calc_end_function=iter_func,
                            )
        point_selector = RandomNodesSelector(nodes_list=nodes_list)
        metric_ipbuild = CoverIndexBuilding(b, ip_val=ip_val_main)
        stop_case_gns = MoreEqualStopCase(target_ip)
        mclp = timing(BestNodesGAKopt(FirstArrivalUnitState(),
                            metric_function=metric_ipbuild,
                            kopt_function = kopt,
                            node_selector = point_selector,
                            population_size=50,
                            elite_size = 15,
                            epochs = 10,  #30
                            mutation_rate = 0.5,
                            mutation_max_count=3,
                            epoch_end_function = epoch_end_function,
                            stop_case_function=stop_case_gns,
        ))
        start_state = {random.sample(list(nodes_list), k=1)[0]:f'псч {i}' for i in range(units_count)}

        bn_times = []
        bn_metrics = []
        bn_units = []
        t()
        for i in range(runs_count):
            t2()
            optimal_nodes, best_metric = mclp(env=g, 
                                    dynamic_nodes=start_state,
                                    area=area_nodes,
                                    )
            bn_times.append(t2(False))
            bn_metrics.append(best_metric)
            bn_units.append(len(optimal_nodes))
            print('\n', i+1, 'Подразделений:', len(optimal_nodes), best_metric, t2(), end='\r')
        time_spent = t()

        # Сохраняем результат вычисления
        times_list[alg_name] = np.mean(bn_times)
        metric_list[alg_name] = np.max(bn_metrics)
        units_count_list[alg_name] = np.min(bn_units)

        save_experiment_state_lscp()
        print('\n')
    print(f'МЕТРИКА: {metric_list[alg_name]}', '| ПОДРАЗДЕЛЕНИЙ', units_count_list[alg_name])
    print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {times_list[alg_name]} сек')


In [ ]:
# from fire_units.metrics import CoverIndexBuilding
# from fire_units.mclp import BestNodesGAKopt
# from genesis.lscp import LSCPCommon
# from genesis.mclp import BestNodesGA, BestNodesKoptG
# from genesis.point_selectors import FarNodeSelector, GenesisNodeSelector, RandomNodesSelector
# from genesis.stop_cases import MoreEqualStopCase, NSameStopCase
# from genesis.utils import TimeCheckPoint


# main_times_list = {}
# main_metrics_list = {}

# units_count = 3

# for district, scope in scopes.items():
#     print(' ')
#     print('='*80)
#     print(district)
#     g = scopes[district][1]
#     b = scopes[district][2]
#     nodes = ox.graph_to_gdfs(g, edges=False)
#     area_nodes = nodes.within(scopes[district][0])
#     nodes_list=set(nodes[area_nodes].index)
#     start_state = {random.sample(list(nodes_list), k=1)[0]:f'псч {i}' for i in range(units_count)}
#     print('Узлов', sum(area_nodes))
#     print('='*80)

#     # функция метрики спроса
#     metric_demand = Demand(b, buildings_demand_field='kfpo_spros')
#     metric_IPB10 = CoverIndexBuilding(b, ip_val=5)

#     # Списки результатов расчета
#     times_list = {}
#     metrics_list = {}


#     # # Расчет GNS
#     # # ============================================================================
#     # print('\n', '# GNS')
#     # print('-'*80)
#     # t = TimeCheckPoint()
#     # # t2 = TimeCheckPoint()

#     # здесь сам расчет
#     def iter_func(i, best_metric, **kwds):
#         print(i, ':', best_metric, end='\r')
#     def mclp_end_function(iteration, best_metric, dynamic_nodes, **kwargs):
#         print(f'Итерация: {iteration} |', f'Лучшая метрика: {best_metric} |', f'Подразделений: {len(dynamic_nodes)}')

#     # bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
#     #                 #    metric_function=metric_demand,
#     #                    metric_function=metric_IPB10,                       
#     #                    )
#     # mclpgns = BestNodesKoptG(state_function=FirstArrivalUnitState(),
#     #                 #    metric_function=metric_demand,
#     #                 #    metric_function=CoverIndex(5),
#     #                    metric_function=metric_IPB10, 
#     #                    best_point_function=bnhc,
#     #                    iterations=5,
#     #                    iter_calc_end_function=iter_func,
#     #                    )

#     # # Случай останоски
#     # stop_case = MoreEqualStopCase(95)

#     # # Селектор следующей точки
#     # # point_selector = RandomNodesSelector()
#     # # point_selector = GenesisNodeSelector(state_function = FirstArrivalUnitState(),
#     # #                                     metric_function = metric_demand)
#     # point_selector = FarNodeSelector(FirstArrivalUnitState())
    
#     # # Сборка алгоритма
#     # genesis = LSCPCommon(mclp_function = mclpgns,
#     #                  point_selector = point_selector,
#     #                 stop_case_function = stop_case,
#     #                 start_names_index = 5,
#     #                 names_pattern='ПСЧ {}',
#     #                 after_mclp_function=mclp_end_function,
#     #                 )
#     # # Собственно применения построенного алгоритма
#     # best_dynamic_nodes, best_metric = genesis(env = g,
#     #                             dynamic_nodes = start_state,
#     #                             area=area_nodes,
#     #                             # static_nodes = existed_units,
#     #                             )

#     # # завершающая стадия расчета
#     # time_spent = t()
#     # print('\n')
#     # print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}')
#     # print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

#     # # Сохраняем результат вычисления
#     # times_list['GNS'] = time_spent
#     # metrics_list['GNS'] = best_metric


#     # Расчет GA+GNS
#     # ============================================================================
#     print('\n', '# GNS')
#     print('-'*80)
#     t = TimeCheckPoint()
#     # t2 = TimeCheckPoint()

#     # здесь сам расчет
#     # # Метрика: ИПЗ-10
#     # IPB10 = CoverIndexBuilding(b)

#     # Функция оптимизации k узлов методом hillclimber
#     bnhc = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
#                         # metric_function = metric_IPB10,
#                         metric_function = metric_demand,
#                         )

#     kopt = BestNodesKoptG(state_function=FirstArrivalUnitState(),
#                         # metric_function = metric_IPB10,
#                         metric_function = metric_demand,
#                         best_point_function=bnhc,
#                         iterations=3,
#                         iter_calc_end_function=iter_func,
#                         )


#     mclp = BestNodesGAKopt(state_function = FirstArrivalUnitState(),
#                         # metric_function = ArrivalTime(),            # Оптимизируем среднее время прибытия в любой узел
#                         metric_function = metric_IPB10,                        # Оптимизируем ИПЗ-10
#                         kopt_function = kopt,
#                         population_size=50,
#                         epochs = 20,
#                         elite_size=15,
#                         epoch_end_function = epoch_end_function,
#                         mutation_max_count = 2,
#                         stop_case_function = NSameStopCase(15)      # Условие остановки не менее N одинаковых метрик
#                         )

#     # Критерий остановки
#     stop_case = MoreEqualStopCase(98)

#     # Селектор следующей точки
#     point_selector = RandomNodesSelector()

#     # Сборка алгоритма
#     genesis = timing(LSCPCommon(mclp_function = mclp,
#                         point_selector = point_selector,
#                         stop_case_function = stop_case,
#                         start_names_index = 12,
#                         names_pattern='псч {}',
#                         after_mclp_function = mclp_end_function,
#                         ))

#     # Расчет только динамические узлы
#     best_dynamic_nodes, best_metric = genesis(env = g,
#                                 dynamic_nodes = start_state,
#                                 # dynamic_nodes = best_dynamic_nodes,     # Продолжение расчета с предыдущего шага                           
#                                 area = area_nodes,
#                                 )

#     # завершающая стадия расчета
#     time_spent = t()
#     print('\n')
#     print(f'ЛУЧШИЙ РЕЗУЛЬТАТ: {best_metric}', 'Подразделений:', len(best_dynamic_nodes))
#     print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {time_spent} сек')

#     # Сохраняем результат вычисления
#     times_list['GNS'] = time_spent
#     metrics_list['GNS'] = best_metric



#     # Сохранение итогов в основные словари:
#     main_times_list[district+' время'] = times_list
#     main_metrics_list[district+' точность'] = metrics_list
    
#     # Промежуточное сохранение в DataFrame с объединением main_times_list и main_metrics_list
#     # ------------------------------
#     tdf = pd.DataFrame({**main_times_list, **main_metrics_list})
#     tdf.to_csv('lscp.csv')
#     tdf.to_csv(f'lscp {district}.csv')

# Черновики

In [ ]:
# Печать карты
def plot_scope_result(g:nx.MultiDiGraph,
                      district:gpd.GeoDataFrame,
                      point:list,
                      buildings:pd.DataFrame,
                      ax=None):
    nodes = ox.graph_to_gdfs(g, edges=False)
    if ax is None:
        fig, ax = ox.plot_graph(g, edge_linewidth=0.2, node_size=0, show=False, bgcolor='white')
    district.boundary.plot(ax=ax, color='r')
    buildings.plot(ax=ax)
    nodes.loc[point].plot(ax=ax, markersize=25, color='red')

    return 1, 2

d = timing(plot_scope_result(g, districts.loc[[district]], [best_node], b))
d

In [ ]:
state, _ = FirstArrivalUnitState()(g, points={best_node:'best'})
area = area_nodes
if isinstance(state, pd.Series):
    state_c = state.loc[area]
else:
    state_c = [x for x in state if x in area]

# Расчет метрики по спросу
merged_df = pd.merge(b,
                    state_c,
                    left_on='node',
                    right_index=True)

merged_df['demand_val'] = merged_df['node'] / merged_df['times']

In [ ]:
def plot_scope_result(g:nx.MultiDiGraph,
                      district:gpd.GeoDataFrame,
                      point:list,
                      buildings:pd.DataFrame,
                      ax=None):
    nodes = ox.graph_to_gdfs(g, edges=False)
    if ax is None:
        fig, ax = ox.plot_graph(g, edge_linewidth=0.2, node_size=0, show=False, bgcolor='white')
    district.boundary.plot(ax=ax, color='r')
    buildings.plot(ax=ax, column='times', cmap=CMAP_GrGldRd, legend=True)
    nodes.loc[point].plot(ax=ax, markersize=25, color='red')

plot_scope_result(g, districts.loc[[district]], [best_node], merged_df)

#### Тест HC с исследованием областей

In [ ]:
bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)

bn_list = []
bn_metrics = []
i=1

size=200
split_list = [list(nodes_list)[i:i+size] for i in range(0, len(nodes_list), size)]

ts = time.time()
while time.time() - ts < etalon_time:
    tcs = time.time()
    best_cur_node = None
    best_cur_metric = 0
    j=1
    for nodes_part in split_list:
        start_node = random.choice(nodes_part)
        best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=start_node)
        if best_metric > best_cur_metric:
            best_cur_node = best_node
            best_cur_metric = best_metric
        print(f'{i}:', j, 'из', len(split_list), best_cur_metric, end='\r')
        j+=1

    bn_list.append(best_cur_node)
    bn_metrics.append(best_cur_metric)
    tcd = time.time() - tcs
    print('\n', i, best_cur_node, best_cur_metric, round(tcd, 2), ' '*10) #, end='\r')
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

In [ ]:
random.choices(list(nodes_list), k=5)

In [ ]:
# from genesis.best_points import NodeMetric
# from genesis.core import StateBase


# class BestNodeBee(BestNodeHillClimbing):
#     '''
#         Поиск лучших узлов графа с использованием алгоритма пчелиной колонии
#         (Artificial Bee Colony Optimization, ABC).
#         Могут быть возвращены только узлы из которых можно попасть в большую часть 
#         других узлов графа.
#         Если таковых узлов нет, возвращается ошибка некорректности графа.
#         (граф должен быть проверен на корректность прежде чем будет передан функции)

#         Узлы для которых метрика не может быть вычислена (например слабо связанные
#         с основным графом) не учитываются.

#         ## Область применения
#         Определение размещения одного узла с наилучшими показателями.
#         Дает достаточно точное решение. Хорошо подходит для больших графов.
#     '''
#     def __init__(self,
#                  state_function: StateBase,
#                  metric_function: MetricBase,
#                  appr_val: float = 0.95,
#                  all_neighbors: bool=True,
#                  scouts_count: int = 100,
#                  best_scouts_count: int = 5,
#                 #  after_local_jump_function: callable = None,
#                  **kwargs) -> None:
#         '''
#         ## Аргументы
#         `state_function`: StateBase
#             функция расчета состояния окружения
#         `metric_function`: MetricBase
#             Функция расчета метрики
#         `appr_val`: = 0.95
#             Доля узлов графа, покрытие которой считается приемлемой для принятия расчетной метрики. 
#             Если при расчете метрик, из стартового узла (узлов) достижимо меньшее количество узлов,
#             то такой узел не рассматривается.
#         `all_neighbors`: bool=True
#             Если True рассматриваются все узлы смежные с рассчитываемым узлом.
#             Если False - только исходящие.
#         `scouts_count`: int=100
#             Количество пчел-разведчиков.
#         `best_scouts_count`: int=5
#             Количество лучших пчел-разведчиков.
#             Для них производится дальнейшая оптимизация.
        
#         '''
#         self.node_metric_func = NodeMetric(state_function,
#                                            metric_function,
#                                            appr_val,
#                                            err_val=None,
#                                            **kwargs)
#         self.scouts_count = scouts_count
#         self.best_scouts_count = best_scouts_count
#         super().__init__(state_function,
#                          metric_function,
#                          appr_val=appr_val,
#                          all_neighbors=all_neighbors,
#                          **kwargs)


#     def __call__(self,
#                  env:nx.MultiDiGraph,
#                  area=None,
#                  nodes_list:set=None,
#                  **kwargs) -> tuple[int | None, float | None]:
#         '''
#         Реализация: `scouts_count` пчел-разведчиков случайным образом проверяют узлы
        
#         при помощи BestNodesHillClimbing ищется лучшая точка, 
#         после чего обезьяна прыгает по ближайшим вершинам и вновь использует BestNodesHillClimbing.

#         Так, пока не будет найден глобальный оптимум или не будет достигнуто количество итераций.

#         ## Аргументы
#         `env`:nx.MultiDiGraph
#             Граф улично-дорожной сети
#         `area`: pd.Series = None
#             Маска узлов графа. Значениями True отмечены узлы графа - цели расчета леса Вороного.
#             Если не указана, расчет производится для всех узлов графа.
#         `nodes_list`:set=None
#             Множество узлов графа которые будут рассмотрены в качестве кандидатов.
#             Если не указан, то будут рассмотрены все узлы графа.

#         ## Возвращает
#         `best_node`: int, `best_metric`: float
#             Лучший узел, лучшая метрика
#         '''

#         if not isinstance(env, nx.MultiDiGraph):
#             raise TypeError("Тип переменной `env` должен быть MultiDiGraph!")
#         if not area is None and not isinstance(area, pd.Series):
#             raise TypeError(f'Аргумент `area` должен иметь тип `pd.Series`! Имеет {type(area)}')

#         # Если списка узлов изначально не передано, рассматриваются все узлы графа
#         if nodes_list is None:
#             nodes_list = env.nodes()

#         # 1. Разведка местности
#         t_list = []
#         t_metric = []
#         for start_node in random.choices(list(nodes_list), k=self.scouts_count):
#             start_metric = self.node_metric_func(env=env, area=area, node=start_node)
#             t_list.append(start_node)
#             t_metric.append(start_metric)
        
#         # 2. Выбираем результаты лучших пчел-разведчиков
#         tdf = pd.DataFrame({'node':t_list, 'metric':t_metric})
#         if self.metric_function.compare(1,2)==2:
#             tdf = tdf.sort_values('metric', ascending=False)
#         else:
#             tdf = tdf.sort_values('metric', ascending=True)
#         # tdf.sort_values('metric', ascending=self.metric_function.compare(1,2)==2)
#         # print(tdf['metric'][:10])

#         # 3. Для каждого из лучших результатов пытаемся найти еще лучшие значения метрики в окрестности
#         best_node = None
#         best_metric  = None
#         for node in tdf['node'][:self.best_scouts_count]:
#             cur_node, cur_metric = super().__call__(
#                     env=env,
#                     start_node=node,
#                     area=area,
#                     **kwargs,
#                     )

#             if best_node is None:
#                 best_node, best_metric = cur_node, cur_metric
#             else:
#                 if best_node != cur_node and \
#                         self.metric_function.compare(best_metric, cur_metric) == cur_metric:
#                     best_node, best_metric = cur_node, cur_metric

#         return best_node, best_metric

In [ ]:
from genesis.best_points import BestNodeBee

bnabc = BestNodeBee(FirstArrivalUnitState(),
                    metric_demand,
                    # scouts_count=150,
                    # best_scouts_count=3,
                    )

ts = time.time()
bn_list = []
bn_metrics = []
i=1
while time.time() - ts < etalon_time:
# while i < 5:
    print(i, end=':')
    tcs = time.time()

    best_node, best_metric = bnabc(env=g, area=area_nodes, nodes_list=nodes_list)

    bn_list.append(best_node)
    bn_metrics.append(best_metric)
    tcd = time.time() - tcs
    print(best_node, best_metric, round(tcd, 2))
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

In [ ]:
from genesis.best_points import NodeMetric


bnhc = BestNodeHillClimbing(FirstArrivalUnitState(), metric_demand)
node_metric = NodeMetric(FirstArrivalUnitState(), metric_demand)


bn_list = []
bn_metrics = []
i=1

k = 100
k_best = 5

ts = time.time()
while time.time() - ts < etalon_time:
# while i < 10:
    tcs = time.time()
    best_cur_node = None
    best_cur_metric = 0

    t_list = []
    t_metric = []
    j=1
    for start_node in random.choices(list(nodes_list), k=k):
        start_metric = node_metric(env=g, area=area_nodes, node=start_node)
        if start_metric > best_cur_metric:
            best_cur_node = start_node
            best_cur_metric = start_metric
        t_list.append(start_node)
        t_metric.append(start_metric)

        print(f'{i}:', j, 'из', k, best_cur_metric, start_metric,  end='\r')
        j+=1

    tdf = pd.DataFrame({'node':t_list, 'metric':t_metric}).sort_values('metric', ascending=False)
    
    best_cur_node = None
    best_cur_metric = 0
    j=1
    for node in tdf['node'][:k_best]:
        best_node, best_metric = bnhc(env=g, area=area_nodes, start_node=node)
        if best_metric > best_cur_metric:
            best_cur_node = best_node
            best_cur_metric = best_metric
        print(f'{i}:', j, 'из', k_best, best_cur_metric, end='\r')
        j+=1

    bn_list.append(best_cur_node)
    bn_metrics.append(best_cur_metric)
    tcd = time.time() - tcs
    print('\n', i, best_cur_node, best_cur_metric, round(tcd, 2), ' '*10) #, end='\r')
    i+=1

tf = time.time()
td = round(tf-ts, 2)
print(f'ВРЕМЯ: {td} сек')

correct_results = bn_list.count(etalon_node)
print(f'КОРРЕКТНОСТЬ: {correct_results} из {len(bn_list)}')
print(f'ВРЕМЯ ОДНОГО РАСЧЕТА: {td/len(bn_list)} сек')

    ВРЕМЯ: 347.34 сек
    КОРРЕКТНОСТЬ: 17 из 20
    ВРЕМЯ ОДНОГО РАСЧЕТА: 17.366999999999997 сек

In [ ]:
size=300
split_list = [list(nodes_list)[i:i+size] for i in range(0, len(nodes_list), size)]
for sl in split_list:
    print(len(sl))